In [ ]:
!pip install -q rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 17.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

def normalize_major(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'[^가-힣A-Za-z0-9]', '', text)
    return text.strip()

def get_closest_major_and_dept_fast(major_name, mapping_majors_list, major_to_dept):
    match, score, _ = process.extractOne(major_name, mapping_majors_list, scorer=fuzz.token_sort_ratio)
    if score >= 80:
        dept = major_to_dept.get(match, None)
        return pd.Series([dept, match, score])
    else:
        return pd.Series([None, None, score])

df_all_years = []

for year in range(2014, 2024):

    # 데이터 불러오기
    df_file = f"학과별_등록금_{year}.csv"
    mapping_file = f"{year}_기준.csv"

    df = pd.read_csv(df_file)
    df.columns = ['기준연도', '학교명', '학과명', '등록금']

    # 결측값 처리
    df['기준연도'] = df['기준연도'].fillna(method='ffill')
    df['학교명'] = df['학교명'].fillna(method='ffill')

    # 형 변환 및 정제
    df['등록금'] = df['등록금'].astype(str).str.replace(',', '', regex=False).astype(float)
    df['기준연도'] = df['기준연도'].astype(int)
    df['학교명'] = df['학교명'].astype(str).str.strip()
    df['학과명'] = df['학과명'].apply(normalize_major)

    df_mapping = pd.read_csv(mapping_file)
    df_mapping['학교명'] = df_mapping['학교명'].astype(str).str.strip()
    df_mapping['학과명'] = df_mapping['학과명'].apply(normalize_major)

    # 단일 학과명 기준 매핑
    major_group = df_mapping.groupby('학과명')['대계열'].nunique()
    single_majors = major_group[major_group == 1].index.tolist()

    df_mapping_single = df_mapping[df_mapping['학과명'].isin(single_majors)].drop_duplicates(subset=['학과명'])[['학과명', '대계열']]
    df_merged = pd.merge(df, df_mapping_single, how='left', on='학과명')

    # 실패한 학과는 학교명+학과명으로 추가 매핑
    df_remaining = df_merged[df_merged['대계열'].isna()].drop(columns=['대계열'])
    df_mapping_multi = df_mapping[~df_mapping['학과명'].isin(single_majors)].drop_duplicates(subset=['학교명', '학과명'])

    df_remaining_merged = pd.merge(df_remaining, df_mapping_multi, how='left', on=['학교명', '학과명'])

    df_combined = pd.concat([
        df_merged[df_merged['대계열'].notna()],
        df_remaining_merged
    ], ignore_index=True)

    # fuzzy matching 대상
    df_unmatched = df_combined[df_combined['대계열'].isna()].copy()

    # 유사도 기반 fuzzy 매칭
    mapping_majors_list = df_mapping['학과명'].dropna().unique().tolist()
    major_to_dept = df_mapping.drop_duplicates('학과명').set_index('학과명')['대계열'].to_dict()

    df_unmatched[['대계열', '추천_학과명', '유사도']] = df_unmatched['학과명'].progress_apply(
        lambda x: get_closest_major_and_dept_fast(x, mapping_majors_list, major_to_dept)
    )

    # 최종 통합
    df_final = pd.concat([
        df_combined[df_combined['대계열'].notna()],
        df_unmatched[df_unmatched['대계열'].notna()],
        df_unmatched[df_unmatched['대계열'].isna()]  # 유사도 실패 항목도 포함
    ], ignore_index=True)

    df_all_years.append(df_final)

# 전체 연도 통합, 결측치 제거
df_total = pd.concat(df_all_years, ignore_index=True)
df_total = df_total.dropna(axis=1, how='all')

# 저장
df_total.to_csv("학과별_등록금_2014_2023_최종.csv", index=False, encoding='utf-8-sig')